# Regression: Wie lange dauert eine VeloCity-Fahrt?

**CRISP-DM-Block, Modeling-Phase (Block 05, Folie „Regression schätzt eine Zahl")**

## Worum es geht

Dies ist das erste der vier Grundverfahren-Notebooks (Regression, Klassifikation, Clustering,
Zeitreihe). Jedes beantwortet eine andere Frage über VeloCity, unseren fiktiven
Fahrradverleih in Würzburg. Hier: **Wie lange dauert eine Fahrt, bevor sie überhaupt
gestartet ist?**

Das ist keine akademische Übung. Wenn VeloCity vorhersagen kann, wie lange ein Rad an
einer Station "besetzt" sein wird, lässt sich die Verfügbarkeit an dieser Station besser
planen.

## Warum Regression und nicht Klassifikation?

Wir wollen eine **Zahl** vorhersagen (Minuten), keine Kategorie. Genau das ist die
Definition von Regression: *"Bei der Regression geht es darum, Zusammenhänge zwischen
Merkmalsausprägungen von Elementen zu finden. Mit einem Regressionsmodell wird eine
abhängige, stetige Merkmalsausprägung durch mehrere unabhängige erklärt."*
(Provost, F., Fawcett, T. (2015): Data Science für Unternehmen, S. 45 f.)

Die **abhängige Variable** (das, was wir vorhersagen wollen) ist hier die Fahrtdauer in
Minuten. Die **unabhängigen Variablen** (das, womit wir sie erklären) sind Merkmale, die
wir schon *vor* Fahrtbeginn kennen: Start-/Zielstation, Wochentag, Uhrzeit, Wetter.

## Woher die Daten kommen

Ganz wichtig, bevor wir anfangen: **`ausleihe.csv` ist erfunden**, aber bewusst mit
realistischen Mustern gebaut (siehe `analytics/README.md` im Repository). `wetter.csv`
dagegen sind **echte historische Wetterdaten** für Würzburg. Wer mit diesen Daten
arbeitet, sollte diesen Unterschied im Kopf behalten — genau wie man es bei echten
Analyseprojekten auch tun sollte, wenn man Daten aus verschiedenen Quellen zusammenführt.

## Lernziele dieses Notebooks

Am Ende könnt ihr:
1. Daten aus mehreren CSV-Dateien sinnvoll zusammenführen (Join)
2. Kategoriale Merkmale für ein Regressionsmodell aufbereiten (One-Hot-Encoding)
3. Einen Trainings-/Testsplit korrekt anlegen und begründen, warum das nötig ist
4. Ein lineares Regressionsmodell trainieren und seine Koeffizienten interpretieren
5. Die Modellgüte mit dem mittleren absoluten Fehler (MAE) bewerten und in eine
   geschäftlich verständliche Aussage übersetzen


## Schritt 1 — Bibliotheken importieren

Wir brauchen `pandas` für die Datentabellen, `numpy` für Zahlenarbeit,
`scikit-learn` für das Modell und `matplotlib` für die Grafik. Das ist der
Standard-Werkzeugkasten für praktisch jede Data-Science-Aufgabe dieser Größenordnung —
dieselben Werkzeuge tauchen in allen vier Notebooks wieder auf.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

pd.set_option("display.max_columns", 20)
print("Bibliotheken geladen.")

## Schritt 2 — Daten laden

Wir laden zwei Tabellen:
- **`ausleihe.csv`**: eine Zeile pro Fahrt — wer, welches Rad, welche Stationen, wann,
  wie lange, wie weit
- **`wetter.csv`**: eine Zeile pro Tag — echte historische Temperatur- und
  Niederschlagswerte für Würzburg

Die Dateien liegen im GitHub-Repository der VeloCity-Fallstudie, `pandas` kann CSV-Dateien
direkt über eine URL laden — das funktioniert genauso in Google Colab wie lokal.

**TODO:** Laden Sie beide Tabellen. Parsen Sie bei `ausleihe.csv` die Spalten `startzeit`
und `endzeit` gleich als Datum/Zeit (Parameter `parse_dates=[...]` bei `pd.read_csv`),
das erspart uns später eine Konvertierung.

In [ ]:
ausleihe = pd.read_csv("https://raw.githubusercontent.com/swrobuts/velocity-fallstudie/main/analytics/ausleihe.csv", parse_dates=["startzeit", "endzeit"])
wetter = pd.read_csv("https://raw.githubusercontent.com/swrobuts/velocity-fallstudie/main/analytics/wetter.csv", parse_dates=["datum"])

print("Fahrten:", len(ausleihe))
print("Wettertage:", len(wetter))
assert len(ausleihe) > 50_000, "Das sind deutlich zu wenige Fahrten - CSV richtig geladen?"
ausleihe.head()

## Schritt 3 — Erst verstehen, dann modellieren

Bevor wir irgendetwas vorhersagen, schauen wir uns an, was wir überhaupt vorhersagen
wollen. Das ist keine Formalität: Wer die Zielgröße nicht kennt, baut blind ein Modell.

**TODO:**
1. Berechnen Sie mit `.describe()` die wichtigsten Kennzahlen der Spalte `dauer_minuten`
   — Moment, die gibt es noch gar nicht! Berechnen Sie sie zuerst selbst aus
   `endzeit - startzeit` (Tipp: die Differenz zweier Zeitstempel in Minuten bekommen Sie
   über `.dt.total_seconds() / 60`).
2. Zeichnen Sie ein Histogramm der Fahrtdauer (`.hist(bins=50)`).

In [ ]:
ausleihe["dauer_minuten"] = ...  # TODO: endzeit - startzeit in Minuten

ausleihe["dauer_minuten"].describe()

In [ ]:
# TODO: Histogramm der Fahrtdauer
...
plt.xlabel("Dauer (Minuten)")
plt.ylabel("Anzahl Fahrten")
plt.title("Verteilung der Fahrtdauer")
plt.show()

**Frage zum Nachdenken (in der Markdown-Zelle darunter beantworten):** Die Verteilung
ist vermutlich rechtsschief — viele kurze Fahrten, wenige sehr lange. Was bedeutet das
für ein lineares Regressionsmodell, das im Mittel den kleinsten quadratischen Fehler
minimiert? Wird es eher die kurzen oder die langen Fahrten gut treffen?

*Ihre Antwort hier …*

## Schritt 4 — Merkmale bauen: Wochentag und Wetter dazuholen

Jetzt kommt der eigentliche Kern der Data Preparation: Wir brauchen Merkmale, die vor
Fahrtbeginn schon bekannt sind. Der Wochentag lässt sich direkt aus `startzeit` ableiten.
Das Wetter kommt aus der zweiten Tabelle und muss über das **Datum** verknüpft werden
(ein sogenannter Join) — das ist derselbe Schritt, der auf der Data-Preparation-Folie in
Block 05 beschrieben ist: Rohdaten werden auf Tagesebene mit externen Quellen
zusammengeführt.

**TODO:**
1. Legen Sie eine Spalte `datum` an (nur der Datumsanteil von `startzeit`, ohne Uhrzeit
   — Tipp: `.dt.normalize()` oder `.dt.date`).
2. Führen Sie `ausleihe` und `wetter` über diese Spalte zusammen (`pd.merge`, `how="left"`).
3. Legen Sie eine Spalte `wochentag` an (`.dt.dayofweek`, 0 = Montag).

In [ ]:
ausleihe["datum"] = ...  # TODO

daten = ...  # TODO: merge von ausleihe und wetter über 'datum'

daten["wochentag"] = ...  # TODO

print(daten.shape)
assert daten["temp_mittel_c"].notna().all(), "Nach dem Join duerfen keine Wetterwerte fehlen!"
daten[["startzeit", "datum", "wochentag", "temp_mittel_c", "niederschlag_mm", "dauer_minuten"]].head()

## Schritt 5 — Kategoriale Merkmale codieren

`start_station_id` ist eine Zahl, aber keine Größe, mit der man rechnen kann — Station 7
ist nicht "größer" als Station 3, es sind einfach unterschiedliche Orte. Ein
Regressionsmodell würde diese Zahl trotzdem als Größenverhältnis missverstehen, wenn wir
sie so belassen.

Die Lösung: **One-Hot-Encoding** — aus einer Spalte mit *k* möglichen Werten werden *k*
neue 0/1-Spalten, jeweils eine pro Station. Genau das Verfahren, das auch auf der
Data-Preparation-Folie in Block 05 genannt wird.

**TODO:** Wenden Sie `pd.get_dummies(...)` auf die Spalte `start_station_id` an und
fügen Sie das Ergebnis den Daten hinzu (`pd.concat`). Hängen Sie direkt `.astype(int)`
an — `get_dummies` liefert sonst `True`/`False`-Werte, und ein Modell rechnet mit 0/1
eindeutiger als mit Wahrheitswerten.

In [ ]:
station_dummies = ...  # TODO: pd.get_dummies auf start_station_id, mit prefix="station", dann .astype(int)
daten = pd.concat([daten, station_dummies], axis=1)

print("Neue Spalten:", list(station_dummies.columns))

## Schritt 6 — Der Trainings-/Testsplit: die wichtigste Regel dieses Notebooks

Bevor wir irgendein Modell trainieren, teilen wir die Daten in zwei Teile:

- **Trainingsdaten** (meist 70–80 %): Damit lernt das Modell.
- **Testdaten** (meist 20–30 %): Die sieht das Modell beim Training nie. Erst danach
  prüfen wir daran, wie gut die Vorhersage auf *neuen*, unbekannten Fahrten funktioniert.

Warum das entscheidend ist: Ein Modell, das an denselben Daten getestet wird, an denen
es trainiert wurde, kann die Trainingsdaten einfach "auswendig lernen" und sieht dann
besser aus, als es wirklich ist. Das nennt man **Overfitting**, und der Testsplit ist das
Standardwerkzeug dagegen — dieselbe Disziplin, die auch beim Zeitreihen-Notebook eine
zentrale Rolle spielt.

**TODO:** Bilden Sie `X` (die Merkmalsspalten: Wochentag, Temperatur, Niederschlag, alle
Stations-Dummy-Spalten) und `y` (die Zielspalte `dauer_minuten`). Teilen Sie beide mit
`train_test_split(X, y, test_size=0.2, random_state=42)` auf.

In [ ]:
merkmalsspalten = ["wochentag", "temp_mittel_c", "niederschlag_mm"] + list(station_dummies.columns)
X = daten[merkmalsspalten]
y = daten["dauer_minuten"]

X_train, X_test, y_train, y_test = ...  # TODO

print("Training:", X_train.shape, "| Test:", X_test.shape)
assert len(X_test) / len(X) == 0.2 or abs(len(X_test) / len(X) - 0.2) < 0.01, "test_size=0.2 pruefen

## Schritt 7 — Das Modell trainieren

Jetzt endlich: das eigentliche Regressionsmodell. `LinearRegression` aus `scikit-learn`
sucht die Gerade (bzw. bei mehreren Merkmalen: die Hyperebene), die den Zusammenhang
zwischen unseren Merkmalen und der Fahrtdauer am besten beschreibt.

**TODO:** Erzeugen Sie ein `LinearRegression()`-Objekt und trainieren Sie es mit
`.fit(X_train, y_train)`.

In [ ]:
modell = ...  # TODO: LinearRegression() erzeugen
...  # TODO: modell.fit(...)

print("Modell trainiert.")

**Die Koeffizienten interpretieren:** Jeder Koeffizient sagt, um wie viele Minuten sich
die vorhergesagte Fahrtdauer ändert, wenn sich das zugehörige Merkmal um eine Einheit
erhöht — alle anderen Merkmale gleich gehalten. Ein positiver Koeffizient bei
`niederschlag_mm` würde zum Beispiel bedeuten: mehr Regen, längere Fahrten (eher
unplausibel) — ein negativer wäre plausibler (Menschen fahren bei Regen zügiger).

In [ ]:
koeffizienten = pd.Series(modell.coef_, index=merkmalsspalten).sort_values()
koeffizienten

## Schritt 8 — Evaluation: Wie gut ist das Modell wirklich?

Auf der Evaluation-Folie in Block 05 steht für Regression: *"Fehlermaß wie MAE,
zurückübersetzt in Minuten oder Kilometer."* Genau das machen wir jetzt.

Der **Mean Absolute Error (MAE)** ist die durchschnittliche absolute Abweichung zwischen
Vorhersage und tatsächlichem Wert — in denselben Einheiten wie die Zielgröße, hier also
in Minuten. Das macht ihn leicht verständlich, auch für ein Publikum ohne
Statistikhintergrund: "Im Schnitt liegt die Vorhersage X Minuten daneben."

**TODO:**
1. Sagen Sie mit `modell.predict(X_test)` die Fahrtdauer für die Testdaten vorher.
2. Berechnen Sie den MAE mit `mean_absolute_error(y_test, vorhersage)`.

In [ ]:
vorhersage = ...  # TODO: modell.predict(X_test)
mae = ...  # TODO: mean_absolute_error(...)

print(f"MAE: {mae:.1f} Minuten")

## Schritt 9 — Rückbezug zur Geschäftsfrage

Ein MAE von wenigen Minuten heißt: Die Vorhersage der Fahrtdauer liegt im Schnitt nur
knapp daneben — nützlich genug, um eine grobe Einschätzung zu geben, wann ein Rad wieder
verfügbar sein wird, aber kein Ersatz für eine Echtzeit-GPS-Ortung.

**Frage zum Nachdenken:** Schauen Sie sich die Koeffizienten aus Schritt 7 noch einmal
an. Welche Stationen haben besonders hohe positive Koeffizienten (also überdurchschnittlich
lange Fahrten)? Passt das zu dem, was wir in Block 05 über Freizeit- vs. Pendlerstationen
gelernt haben? Das ist bereits ein kleiner Vorgeschmack auf das nächste Notebook
(Clustering) — dort gruppieren wir Stationen systematisch nach genau solchen Mustern,
statt sie einzeln zu betrachten.

*Ihre Antwort hier …*

## Zusammenfassung

In diesem Notebook haben Sie:
- zwei Datenquellen (eine fiktive, eine echte) über ein gemeinsames Datum verknüpft,
- eine kategoriale Variable per One-Hot-Encoding modellfähig gemacht,
- einen sauberen Trainings-/Testsplit angelegt und begründet,
- ein lineares Regressionsmodell trainiert, interpretiert und mit dem MAE bewertet.

**Weiter geht's mit Notebook 2 — Klassifikation:** Dort sagen wir keine Zahl mehr voraus,
sondern eine Kategorie: Braucht ein Rad bald Wartung, ja oder nein?